# 01 — Architecture tour of `gepa`, in the same kernel that runs it

A coding agent that just ran `gepa.optimize` and got back a result usually wants to know *how* the optimizer reached that result — which selector picked the candidate, which proposer rewrote the prompt, what the engine main loop actually does.

This notebook is pure code archaeology — no LLM calls, no budget. The advantage on display: `inspect.getsource` walks the live module that just produced the previous notebook's `GEPAResult`. The same kernel that runs the optimizer is the kernel that browses its source.

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/01-gepa.md` rank #1, "trace-inspection ergonomics."

## 1. Where is the installed code?

The installed `gepa` package — the same one the previous notebook called — lives somewhere on disk. Find it first so every later cell references real files.

In [1]:
import gepa, os, inspect

gepa_root = os.path.dirname(gepa.__file__)
print("gepa package root:", gepa_root)
print()
# What modules live under it?
for root, dirs, files in os.walk(gepa_root):
    if "__pycache__" in root:
        continue
    depth = root.replace(gepa_root, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{os.path.basename(root) or '(gepa)'}/")
    for f in sorted(files):
        if f.endswith(".py") and f != "__init__.py":
            print(f"{indent}  {f}")
    if depth >= 2:
        dirs.clear()

gepa package root: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa

gepa/
  api.py
  gepa_utils.py
  image.py
  optimize_anything.py
  visualization.py
  core/
    adapter.py
    callbacks.py
    data_loader.py
    engine.py
    result.py
    state.py
  strategies/
    batch_sampler.py
    candidate_selector.py
    component_selector.py
    eval_policy.py
    instruction_proposal.py
  utils/
    code_execution.py
    stdio_capture.py
    stop_condition.py
  proposer/
    base.py
    merge.py
    reflective_mutation/
      base.py
      reflective_mutation.py
  adapters/
    dspy_adapter/
      dspy_adapter.py
      instruction_proposal.py
    mcp_adapter/
      mcp_adapter.py
      mcp_client.py
    dspy_full_program_adapter/
      dspy_program_proposal_signature.py
      full_program_adapter.py
    anymaths_adapter/
      anymaths_adapter.py
    default_adapter/
      default_adapter.py
    generic_rag_adapter/
      evaluation_metrics.py
      generic_rag

## 2. The `GEPAAdapter` protocol — the boundary between GEPA and your system

GEPA optimizes any text artifact through an adapter. Every custom adapter (DefaultAdapter, DSPy, MCP, RAG, …) implements this protocol. Look at the actual abstract base:

In [2]:
from gepa.core import adapter as adapter_mod

src = inspect.getsource(adapter_mod.GEPAAdapter)
# Trim to the class definition + abstract method signatures
lines = src.splitlines()
print(f"file: {inspect.getfile(adapter_mod.GEPAAdapter)}")
print(f"total lines in class: {len(lines)}")
print()
print("\n".join(lines[:80]))

file: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/adapter.py
total lines in class: 123

class GEPAAdapter(Protocol[DataInst, Trajectory, RolloutOutput]):
    """
    GEPAAdapter is the single integration point between your system
    and the GEPA optimization engine. Implementers provide three responsibilities:

    The following are user-defined types that are not interpreted by GEPA but are used by the user's code
        to define the adapter:
    DataInst: User-defined type of input data to the program under optimization.
    Trajectory: User-defined type of trajectory data, which typically captures the
        different steps of the program candidate execution.
    RolloutOutput: User-defined type of output data from the program candidate.

    The following are the responsibilities of the adapter:
    1) Program construction and evaluation (evaluate):
       Given a batch of DataInst and a "candidate" program (mapping from named components
  

## 3. The engine main loop

`GEPAEngine.run()` is where the optimization actually happens — propose, evaluate, accept-or-reject, repeat. Read the live source:

In [3]:
from gepa.core import engine as engine_mod

print(f"file: {inspect.getfile(engine_mod)}")
print(f"public symbols: {[n for n in dir(engine_mod) if not n.startswith('_')][:15]}")
print()

# Print just the engine's main loop
src = inspect.getsource(engine_mod.GEPAEngine.run)
lines = src.splitlines()
print(f"GEPAEngine.run — {len(lines)} lines")
print("=" * 70)
print(src[:3500] + ("\n..." if len(src) > 3500 else ""))

file: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/core/engine.py
public symbols: ['Any', 'BudgetUpdatedEvent', 'CandidateAcceptedEvent', 'CandidateRejectedEvent', 'DataId', 'DataInst', 'DataLoa
der', 'ErrorEvent', 'EvaluationCache', 'EvaluationPolicy', 'ExperimentTracker', 'FrontierType', 'FullEvaluationPolicy', 'GEPAAda
pter', 'GEPACallback']

GEPAEngine.run — 399 lines
    def run(self) -> GEPAState[RolloutOutput, DataId]:
        # Check tqdm availability if progress bar is enabled
        progress_bar = None
        if self.display_progress_bar:
            if tqdm is None:
                raise ImportError("tqdm must be installed when display_progress_bar is enabled")

            # Check if stop_callback contains MaxMetricCallsStopper
            total_calls: int | None = None
            stop_cb = self.stop_callback
            if stop_cb is not None:
                max_calls_attr = getattr(stop_cb, "max_metric_calls", None)
                if i

In [4]:
# Find the iteration loop inside engine.run — print the middle section that contains the propose/accept logic
src_lines = src.splitlines()
# Show the loop body — look for the 'while' or 'for iteration' pattern
loop_start = next(
    (i for i, l in enumerate(src_lines) if "while" in l and "stop" in l.lower()), None
)
if loop_start is None:
    loop_start = next(
        (i for i, l in enumerate(src_lines) if l.strip().startswith("while ")), 0
    )
print(f"loop starts ~line {loop_start} of run()")
print("=" * 70)
print("\n".join(src_lines[loop_start : loop_start + 60]))

loop starts ~line 166 of run()
        while not self._should_stop(state):
            if self.display_progress_bar and progress_bar is not None:
                delta = state.total_num_evals - last_pbar_val
                progress_bar.update(delta)
                last_pbar_val = state.total_num_evals

            assert state.is_consistent()
            proposal_accepted = False
            iteration_started = False
            try:
                state.save(self.run_dir, use_cloudpickle=self.use_cloudpickle)
                notify_callbacks(
                    self.callbacks,
                    "on_state_saved",
                    StateSavedEvent(
                        iteration=state.i + 1,
                        run_dir=self.run_dir,
                    ),
                )

                state.i += 1
                state.full_program_trace.append({"i": state.i})

                # Notify callbacks of iteration start
                notify_callbacks(
                   

## 4. The reflective mutation proposer — what reflection looks like as code

The "Reflective" in GEPA-Reflective Mutation is the LLM call that reads an evaluation batch and proposes a better candidate. Look at `propose()`:

In [5]:
from gepa.proposer.reflective_mutation import reflective_mutation as rm_mod

print("file:", inspect.getfile(rm_mod))
classes = [n for n in dir(rm_mod) if "Proposer" in n or "Reflect" in n]
print("propose-related classes:", classes)
print()

# Show the propose method signature + first ~50 lines
RM = rm_mod.ReflectiveMutationProposer
sig = inspect.signature(RM.propose)
print("ReflectiveMutationProposer.propose signature:")
print(" ", sig)
print()

# Method body
body = inspect.getsource(RM.propose)
print(f"propose body — {len(body.splitlines())} lines")
print("=" * 70)
print(body[:2500] + ("\n..." if len(body) > 2500 else ""))

file: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/proposer/reflective_mutation/reflective_mutat
ion.py
propose-related classes: ['ReflectionComponentSelector', 'ReflectiveDatasetBuiltEvent', 'ReflectiveMutationProposer']

ReflectiveMutationProposer.propose signature:
  (self, state: gepa.core.state.GEPAState) -> gepa.proposer.base.CandidateProposal | None

propose body — 248 lines
    def propose(self, state: GEPAState) -> CandidateProposal | None:
        i = state.i + 1

        curr_prog_id = self.candidate_selector.select_candidate_idx(state)
        curr_prog = state.program_candidates[curr_prog_id]
        state.full_program_trace[-1]["selected_program_candidate"] = curr_prog_id
        self.logger.log(
            f"Iteration {i}: Selected program {curr_prog_id} score: {state.program_full_scores_val_set[curr_prog_id]}"
        )

        # Notify candidate selected
        notify_callbacks(
            self.callbacks,
            "on_candidate_se

## 5. Strategies — the swappable parts

GEPA's behavior is steered by four pluggable strategies. Each is a small class with a clear interface — see them live:

In [6]:
from gepa.strategies import candidate_selector as cs_mod
from gepa.strategies import batch_sampler as bs_mod
from gepa.strategies import component_selector as comp_mod


def short(cls):
    """Print one-line description + key method signature."""
    doc = (cls.__doc__ or "").strip().split("\n")[0]
    methods = [
        m for m in dir(cls) if not m.startswith("_") and callable(getattr(cls, m))
    ]
    return f"  - {cls.__name__:<36} :: {doc[:80] if doc else '(no docstring)'}"


print("CandidateSelector implementations:")
for name in [
    "ParetoCandidateSelector",
    "CurrentBestCandidateSelector",
    "EpsilonGreedyCandidateSelector",
    "TopKParetoCandidateSelector",
]:
    cls = getattr(cs_mod, name, None)
    if cls:
        print(short(cls))

print("\nBatchSampler implementations:")
for name in dir(bs_mod):
    cls = getattr(bs_mod, name)
    if isinstance(cls, type) and "Sampler" in name and not name.startswith("_"):
        print(short(cls))

print("\nComponentSelector implementations:")
for name in dir(comp_mod):
    cls = getattr(comp_mod, name)
    if isinstance(cls, type) and "Component" in name and "Selector" in name:
        print(short(cls))

CandidateSelector implementations:
  - ParetoCandidateSelector              :: (no docstring)
  - CurrentBestCandidateSelector         :: (no docstring)
  - EpsilonGreedyCandidateSelector       :: (no docstring)
  - TopKParetoCandidateSelector          :: Pareto selection restricted to the top K programs by aggregate score.

BatchSampler implementations:
  - BatchSampler                         :: (no docstring)
  - EpochShuffledBatchSampler            :: Mirrors the original batching logic:

ComponentSelector implementations:
  - AllReflectionComponentSelector       :: (no docstring)
  - ReflectionComponentSelector          :: (no docstring)
  - RoundRobinReflectionComponentSelector :: (no docstring)


### The default Pareto selector — its actual selection logic

Why is candidate selection non-trivial? Because the "best" candidate depends on which task you care about — different candidates can dominate different tasks. The `ParetoCandidateSelector` picks from the Pareto frontier each round to avoid local optima.

In [7]:
src = inspect.getsource(cs_mod.ParetoCandidateSelector)
print(src[:2200] + ("\n..." if len(src) > 2200 else ""))

class ParetoCandidateSelector(CandidateSelector):
    def __init__(self, rng: random.Random | None):
        if rng is None:
            self.rng = random.Random(0)
        else:
            self.rng = rng

    def select_candidate_idx(self, state: GEPAState) -> int:
        assert len(state.program_full_scores_val_set) == len(state.program_candidates)
        return select_program_candidate_from_pareto_front(
            state.get_pareto_front_mapping(),
            state.per_program_tracked_scores,
            self.rng,
        )


In [8]:
# Where does the Pareto selection actually happen? Dig one level deeper —
# the helper that the selector delegates to.
from gepa import gepa_utils

fn = getattr(gepa_utils, "select_program_candidate_from_pareto_front", None)
if fn is not None:
    print(f"file: {inspect.getfile(fn)}")
    print()
    print(inspect.getsource(fn))

file: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa/gepa_utils.py

def select_program_candidate_from_pareto_front(
    pareto_front_programs: Mapping[Any, set[int]],
    train_val_weighted_agg_scores_for_all_programs: list[float],
    rng: random.Random,
) -> int:
    train_val_pareto_front_programs = pareto_front_programs
    new_program_at_pareto_front_valset = remove_dominated_programs(
        train_val_pareto_front_programs, scores=train_val_weighted_agg_scores_for_all_programs
    )
    program_frequency_in_validation_pareto_front = {}
    for testcase_pareto_front in new_program_at_pareto_front_valset.values():
        for prog_idx in testcase_pareto_front:
            if prog_idx not in program_frequency_in_validation_pareto_front:
                program_frequency_in_validation_pareto_front[prog_idx] = 0
            program_frequency_in_validation_pareto_front[prog_idx] += 1

    sampling_list = [
        prog_idx for prog_idx, freq in program_fr

## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. Where is the installed code?
- 2. The `GEPAAdapter` protocol — the boundary between GEPA and your system
- 3. The engine main loop
- 4. The reflective mutation proposer — what reflection looks like as code
- 5. Strategies — the swappable parts
- 6. What the kernel gave us, that a script-based tour can't

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import json as _json
from pathlib import Path as _Path
import collections as _c

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/gepa/01-architecture-tour.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print(f"output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")


## 6. What the kernel gave us, that a script-based tour can't

This whole notebook was source-only — zero LLM calls, zero budget. But it could not run as a one-shot script because the value is **in being able to ask follow-up questions without re-importing or re-reading the file tree**:

- `inspect.getsource` was the workhorse — it works because `gepa` is loaded in memory.
- The tree walk, the protocol class, the engine loop, the proposer, the strategies all share the same import — same in-memory module graph.
- If we wanted to step into one of these strategies and pickle a synthetic `GEPAState` to feed it, the previous notebook's `result` is one cell away — *no rerun of the AIME loader, no re-credentialing*.

The kernel turned the GEPA package into a live thing we can navigate, not a static directory to grep.

## Data sources

| Source | Path |
|---|---|
| GEPA package on disk | `~/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/gepa` |
| Upstream source mirror | `~/Documents/GitHub/gepa` (cloned from `github.com/gepa-ai/gepa`) |
| `gepa` PyPI version | `0.1.1` |

Every `inspect.getsource(...)` block above shows real file contents from the installed package — no transcription, no paraphrase.

→ **Next:** [`02-default-adapter-deep-dive.ipynb`](02-default-adapter-deep-dive.ipynb) — exercise the DefaultAdapter end-to-end and read traces as a DataFrame.